# Bout discretisation

From the rig's raw files to the table of bouts that every model is trained on (`data/bouts_clean.csv`): first one
table per fish at 30 frames per second (`data/frames/`, see `data/README.md`) with the fish's position and
orientation and the position of every dot in every frame, then the bouts.

In [ ]:
import io

import numpy as np
import pandas as pd
from figures import data_figures, fig_bout_trajectory
from twin import cleaning, frames as FR, paths
from twin import segment_bouts as SB
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

## The 30 fps tables
Built from the rig's three raw files (`data/raw/`): the positions file (every fish's pose per frame), the
trajectory file (each dot's offset from the fish at stimulus onset) and the dish geometry. A dot is placed in the
dish from the fish's pose at the first frame of its stimulus block; the table starts one pause before the first
stimulus. The tables are written to `data/frames/` only if that folder is empty.

In [ ]:
built = FR.build(*FR.read_raw())
if not any(paths.FRAMES.glob("*_transformed_abs.csv")):
    FR.write(built)
frames_raw = [(int(p.stem.split("_f")[-1].split("_")[0]), pd.read_csv(p))
              for p in sorted(paths.FRAMES.glob("*_transformed_abs.csv"))]
labels = lambda t: [c for c in t.columns if c == "stimulus" or c.endswith("_stim")]
diff = max(np.nanmax(np.abs(built[f"f{k}"].drop(columns=labels(t)).to_numpy(float)
                            - t.drop(columns=labels(t)).to_numpy(float))) for k, t in frames_raw)
same = all((built[f"f{k}"][labels(t)].astype(str).to_numpy() == t[labels(t)].astype(str).to_numpy()).all()
           for k, t in frames_raw)
print(f"{len(frames_raw)} fish, {len(frames_raw[0][1]):,} frames each ({len(frames_raw[0][1]) / 30 / 60:.0f} min); "
      f"rebuilt from the raw files: labels identical {same}, positions within {diff:.0e} px")
frames_raw[0][1].dropna().head()

## Bout detection (Figure fig:detection)
A bout is a peak of the smoothed swimming speed above its mean plus one SD, at least 0.4 s after the previous one,
whose displacement around the peak is between 0.3 and 25 mm; its onset is the speed minimum before the peak.

In [ ]:
data_figures.fig_detection(data_figures.load())

## From the trajectory to bouts (Figure fig:bouttraj)

In [ ]:
fig_bout_trajectory.main()

## Every fish segmented
The segmenter on the 30 fps tables gives the bout table `data/bouts_GsizeMotionSingle.csv`. Turn, displacement and
interval of a bout are measured from its onset to the next one.

In [ ]:
raw = cleaning.segment(frames_raw).drop(columns=["near_wall", "v_mm_s"])
archived = pd.read_csv(paths.BOUTS_RAW)
cols = [c for c in archived.columns if c in raw.columns]
print(f"{len(raw):,} bouts, {raw.fish_id.nunique()} fish")
print("identical to data/bouts_GsizeMotionSingle.csv:",
      len(raw) == len(archived) and all(np.allclose(raw[c], archived[c], equal_nan=True)
                                        if pd.api.types.is_numeric_dtype(archived[c])
                                        else (raw[c].astype(str) == archived[c].astype(str)).all() for c in cols))

## Step 1: tracking excursions repaired
The tracker sometimes loses the fish for one to five frames and reports it elsewhere, generally at the dish edge,
which the speed smoothing turns into impossibly fast bouts. Such excursions (a step over 100 mm/s that comes back
within five frames) are bridged by linear interpolation.

In [ ]:
frames, report, repaired = cleaning.repair_frames()
print(f"repaired {report.repaired.sum()} of {report.frames.sum():,} frames")
report

## Step 2: re-segmented, gaps flagged, model columns added
A bout whose interval exceeds 10 s spans a quiescent gap: its step and turn are measured across the gap. A bout is
*active* if neither its own leg nor the previous one spans a gap; models are fitted on active bouts only. Then the
block clocks, the runs between gaps and the lags within a run, the dot and wall bearings in the travel frame, and the
position of every dot at each onset.

In [ ]:
bouts = cleaning.add_dot_positions(cleaning.add_model_columns(cleaning.flag_gaps(cleaning.segment(frames))))
a = bouts[bouts.active]
print(f"{len(bouts):,} bouts, {len(a):,} active ({a.groupby('fish_id').size().min():,}-"
      f"{a.groupby('fish_id').size().max():,} per fish); {bouts.gap_after.sum():,} bouts end in a gap > 10 s; "
      f"{bouts.run_id.nunique():,} runs")

## The training table
Written to `data/bouts_clean.csv` if it does not exist yet, else compared with it.

In [ ]:
if not paths.BOUTS.exists():
    bouts.to_csv(paths.BOUTS, index=False)
    repaired.to_csv(paths.REPAIRED, index=False)
    print("wrote", paths.BOUTS, "and", paths.REPAIRED)
else:
    old = pd.read_csv(paths.BOUTS)
    new = pd.read_csv(io.StringIO(bouts.to_csv(index=False)))      # the same CSV round trip
    same = list(old.columns) == list(new.columns) and len(old) == len(new) and all(
        np.allclose(old[c], new[c], equal_nan=True) if pd.api.types.is_numeric_dtype(old[c])
        else (old[c].astype(str) == new[c].astype(str)).all() for c in old.columns)
    print(f"identical to {paths.BOUTS.name} ({len(old):,} rows, {old.shape[1]} columns):", same)
    print("repaired frames identical:", repaired.reset_index(drop=True).equals(pd.read_csv(paths.REPAIRED)))